2D Guass Law (Laplace equation and Guass equation)

In [ ]:
import os
import warnings
#import subprocess
from pathlib import Path

import sympy as sp
from sympy import Symbol, Function, Heaviside

import physicsnemo.sym
from physicsnemo.sym.hydra import to_absolute_path, instantiate_arch, PhysicsNeMoConfig
from physicsnemo.sym.solver import Solver
from physicsnemo.sym.domain import Domain
from physicsnemo.sym.geometry.primitives_2d import Rectangle
from physicsnemo.sym.domain.constraint import (
    PointwiseBoundaryConstraint,
    PointwiseInteriorConstraint,
    PointwiseConstraint
)

from physicsnemo.sym.domain.validator import PointwiseValidator
from physicsnemo.sym.domain.inferencer import PointwiseInferencer
from physicsnemo.sym.key import Key
from physicsnemo.sym.utils.io import (
    csv_to_dict,
    ValidatorPlotter,
    InferencerPlotter,
)

from physicsnemo.sym.amp import AmpManager
from physicsnemo.sym.eq.pde import PDE

In [ ]:
import numpy as np
#epsilon_0 = 8.854187817e-12  # vacuum permittivity in F/m
#electron_charge = 1.602176634e-19  # elementary charge in C. Take positive sign


height, width = 1, 1
dx, dy = 0.005, 0.005
x0, y0 = -width / 2, -height / 2

charge_radius = 0.1
point_charges_positions= np.array([[0.3, 0.3], [0.7, 0.7]]) + np.array([[x0, y0]])      # dim: [2, 2] + [1, 2] -> [2, 2]
charges = np.array([2, -2]) # charge to epsilon ratio instead of charge. 
rho = None

# boundary conditions
bottom_boundary = 0.8
top_boundary = 0.8
left_boundary = 0
right_boundary = 0

iterations = 20001



In [ ]:
class LaplaceEquation2D(PDE):
    name = "LaplaceEquation2D"

    def __init__(self):
        # coordinates
        x, y = Symbol("x"), Symbol("y")

        input_variables ={"x": x, "y": y}

        # electric potential function
        v = Function("v")(*input_variables)

        # set equation
        self.equations = {}
        self.equations["laplace"] = v.diff(x, 2) + v.diff(y, 2)


class GaussEquation2D(PDE):
    name = "GaussEquation2D"

    def __init__(self):
        # coordinates
        x, y = Symbol("x"), Symbol("y")

        input_variables ={"x": x, "y": y}

        # electric potential function
        v = Function("v")(*input_variables)

        rho = 0
        for i in range(len(charges)):
            rho += charges[i] / (sp.pi * charge_radius**2) * (Heaviside(charge_radius - sp.sqrt((x - point_charges_positions[i, 0])**2 + (y - point_charges_positions[i, 1])**2)))

        # set equation
        self.equations = {}
        self.equations["gauss"] = v.diff(x, 2) + v.diff(y, 2) + rho         # charge density to epsilon ratio instead of charge density.

In [ ]:
#subprocess.run("mkdir -p ./conf", shell=True, check=True)
Path("./conf").mkdir(parents=True, exist_ok=True)
with open('./conf/config.yaml', 'w') as f:
    f.write(
        f"""
        defaults:
            - physicsnemo_default
            - arch:
                - fully_connected
            - scheduler: tf_exponential_lr
            - optimizer: lamb
            - loss: sum
            - _self_
        
        scheduler:
            decay_rate: 0.95
            decay_steps: 4000
        
        training:
            rec_validation_freq: 1000
            rec_inference_freq: 2000
            rec_monitor_freq: 1000
            rec_constraint_freq: 2000
            max_steps: 30000

        batch_size:
            SideBoundary: 2000
            TopBottomBoundary: 2000
            Interior: 9000
            Training: 1000

        graph:
            func_arch: true

        # for jupyter notebook only
        hydra:
            job:
                name: jupyter_Gauss-PINNs-data
        """
    )

In [ ]:
# For jupyter notebook only
import sys
sys.argv = ["program"]

root_dir = os.getcwd()

@physicsnemo.sym.main(config_path=root_dir + "/conf", config_name="config")
def run(cfg: PhysicsNeMoConfig) -> None:
    #diff_equation = LaplaceEquation2D()
    diff_equation = GaussEquation2D()
    potential_net = instantiate_arch(
        input_keys = [Key("x"), Key("y")], 
        output_keys = [Key("v")],
        cfg = cfg.arch.fully_connected,
    )
    nodes = diff_equation.make_nodes() + [potential_net.make_node(name="potential_network")]

    # add constraints to solver
    # make geometry
    width, height = 1, 1
    x, y = Symbol("x"), Symbol("y")
    rec = Rectangle((-width / 2, -height / 2), (width / 2, height / 2))

    # make domain
    domain = Domain()

    # load your training data CSV
    train_file_path = "2point_charges-Jacobi.csv"

    if os.path.exists(to_absolute_path(train_file_path)):
        mapping = {"x": "x", "y": "y", "v": "v"}
        train_var = csv_to_dict(to_absolute_path(train_file_path), mapping)
        invar_train = {key: value for key, value in train_var.items() if key in ["x", "y"]}
        outvar_train = {key: value for key, value in train_var.items() if key in ["v"]}

        # add training data constraint
        training_data_constraint = PointwiseConstraint.from_numpy(
            nodes = nodes,
            invar = invar_train,
            outvar = outvar_train,
            batch_size = cfg.batch_size.Training,
        )
        domain.add_constraint(training_data_constraint, "training_data_constraint")
    

    # initial condition
    interior = PointwiseInteriorConstraint(
        nodes = nodes, 
        geometry = rec,
        #outvar = {"laplace": 0},
        outvar = {"gauss": 0},
        batch_size = cfg.batch_size.Interior,
    )
    domain.add_constraint(interior, "interior")

    side_boundary = PointwiseBoundaryConstraint(
        nodes = nodes,
        geometry = rec,
        outvar = {"v": 0.0},
        batch_size = cfg.batch_size.SideBoundary,
        criteria = ((y < height / 2) & (y > -height / 2))
    )
    domain.add_constraint(side_boundary, "side_boundary")

    top_bottom_boundary = PointwiseBoundaryConstraint(
        nodes = nodes,
        geometry = rec,
        outvar = {"v": 0.8},
        batch_size = cfg.batch_size.TopBottomBoundary,
        criteria = (x < width / 2) & (x > -width / 2)
    )
    domain.add_constraint(top_bottom_boundary, "top_bottom_boundary")


    # add validator
    #file_path = "Laplace-Jacobi.csv"
    #file_path = "2point_charges-Jacobi.csv"
    file_path = train_file_path
    if os.path.exists(to_absolute_path(file_path)):
        #mapping = {"x": "x", "y": "y", "v": "v"}
        #var = csv_to_dict(to_absolute_path(file_path), mapping)
        # invar_numpy = {key: value for key, value in var.items() if key in ["x", "y"]}
        # outvar_numpy = {key: value for key, value in var.items() if key in ["v"]}
        invar_numpy = invar_train
        outvar_numpy = outvar_train

        validator = PointwiseValidator(
            nodes = nodes,
            invar = invar_numpy,
            true_outvar = outvar_numpy,
            batch_size = 1024,
            plotter = ValidatorPlotter(),
        )
        domain.add_validator(validator)

        # add inferencer
        inferencer = PointwiseInferencer(
            nodes = nodes,
            invar = invar_numpy,
            output_names = ["v"],
            batch_size = 1024,
            plotter = InferencerPlotter(),
        )
        domain.add_inferencer(inferencer, "inf_data")

    else:
        warnings.warn(f"Directory {file_path} does not exist.")
        
    # make solver
    slv = Solver(cfg, domain)

    # start solver
    slv.solve()

if __name__ == "__main__":
    run()


In [ ]:
# Laplace Equation; Gauss-Seidel Method (CPU-version)
import numpy as np
import time
from tqdm import trange
import matplotlib.pyplot as plt

start_time = time.perf_counter()

height, width = 1, 1
dx, dy = 0.005, 0.005
x0, y0 = -width / 2, -height / 2

iterations = 200001

def boundary_conditions(nx, ny):     # interval x, interval y, u(x,y)
    """Set up boundary conditions for the potential field."""
    u = np.zeros((ny, nx))  # y and x are reversed due to the array indexing;
    
    # Remeber the coorinate of y is up-side down
    u[0, :] = 1     # real bottom boundary
    u[-1, :] = 1    # real top boundary
    u[:, 0] = 0     # left boundary
    u[:, -1] = 0    # right boundary

    return u

def Jacobi(u, max_iterations):
    """Perform Jacobi iterations to solve the Laplace equation."""
    #ny, nx = u.shape
    u_new = u.copy()
    for iteration in trange(max_iterations, desc="Jacobi", unit="iter"):
        u_new[1:-1, 1:-1] = 0.25 * (u[2:, 1:-1] +u[:-2, 1:-1] +u[1:-1, 2:] +u[1:-1, :-2])
        
        # mimic the periodic boundary condition
        # u_new[1:-1, 0] = 0.25 * (u[2:, 0] + u[:-2, 0] + u[1:-1, 1] + u[1:-1, -1]) 
        # u_new[1:-1, -1] = 0.25 * (u[2:, -1] + u[:-2, -1] + u[1:-1, 0] + u[1:-1, -2])  

        #u_new[0:ordery_b1, orderx_b0:orderx_b1] = 0     # building block boundary condition
        u = u_new
    print(f'Final iteration: {iteration}, Residual: {np.linalg.norm(u_new - u)}')

    return u

def main():
    nx = int(width / dx + 1)
    ny = int(height / dy + 1)
    print (nx, ny) 
    v = boundary_conditions(nx, ny)
    v = Jacobi(v, iterations)    # u, max_iterations

    x = np.linspace(x0, x0 + width, nx)
    y = np.linspace(y0, y0 + height, ny)
    X, Y = np.meshgrid(x, y)
    plt.contourf(X, Y, v, levels=50, cmap='viridis')
    plt.colorbar(format='%.3g')
    plt.title('v')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.savefig("Laplace-Jacobi.png", dpi=600)
    plt.show()

    structure_name = "Laplace-Jacobi.csv"
    with open(structure_name, "w") as f:
        f.write("v, x, y\n")
        for i in range(int(ny)):
            for j in range(int(nx)):
                f.write(f"{v[i, j]}, {x[j]}, {y[i]}\n")


if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
x0, y0 = -0.5, -0.5
height, width = 1, 1
nx, ny = 200, 200
dx, dy = 1 / (nx - 1), 1 / (ny - 1)
charge_radius = 0.1
point_charges_positions= np.array([[0.25, 0.25], [0.75, 0.75]]) + np.array([[x0, y0]])      # dim: [2, 2] + [1, 2] -> [2, 2]
charges = np.array([1, -1])

rho = np.zeros((ny, nx))  # charge density array
charges_density = charges #/ (np.pi *charge_radius ** 2) # testing

point_charges = np.array([[positions[0], positions[1], charge_density] for positions, charge_density in zip(point_charges_positions, charges_density)])
print(point_charges)
#n_charges = []
for point_charge in point_charges:
    # i and j are following the indexing of the array, which is different from the x-y coordinate system
    for i in range(ny):
        for j in range(nx):
            r = np.sqrt((x0 + j * dx - point_charge[0])**2 + (y0 + i * dy - point_charge[1])**2)
            if r <= charge_radius:  # include a small tolerance to account for numerical errors
                #n_charges.append((i, j))
                rho[i, j] = point_charge[-1]
#print(rho)

x = np.linspace(x0, x0 + width, nx)
y = np.linspace(y0, y0 + height, ny)
X, Y = np.meshgrid(x, y)
plt.contourf(X, Y, rho, levels=50, cmap='viridis')
plt.colorbar(format='%.3g')
plt.title('rho')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

In [ ]:
import numpy as np
import time
from tqdm import trange
import matplotlib.pyplot as plt

In [ ]:
# hyperparameters
#epsilon_0 = 8.854187817e-12  # vacuum permittivity in F/m
#electron_charge = 1.602176634e-19  # elementary charge in C. Take positive sign

height, width = 1, 1
dx, dy = 0.0025, 0.0025
x0, y0 = -width / 2, -height / 2

charge_radius = 0.1
point_charges_positions= np.array([[0.3, 0.3], [0.7, 0.7]]) + np.array([[x0, y0]])      # dim: [2, 2] + [1, 2] -> [2, 2]
charges = np.array([2, -2]) # charge to epsilon ratio instead of charge. 
rho = None

# boundary conditions
bottom_boundary = 0.8
top_boundary = 0.8
left_boundary = 0
right_boundary = 0

iterations = 200001

In [ ]:
def charge_density(nx, ny, point_charges_positions, charges, charge_radius):
    rho = np.zeros((ny, nx))  # charge density array
    charges_density = charges / (np.pi * charge_radius ** 2)

    point_charges = np.array([[positions[0], positions[1], charge_density] for positions, charge_density in zip(point_charges_positions, charges_density)])
    for point_charge in point_charges:
        # i and j are following the indexing of the array, which is different from the x-y coordinate system
        for i in range(ny):
            for j in range(nx):
                r = np.sqrt((x0 + j * dx - point_charge[0])**2 + (y0 + i * dy - point_charge[1])**2)
                if r <= charge_radius:  # include a small tolerance to account for numerical errors
                    rho[i, j] = point_charge[-1]
    
    return rho

In [ ]:
# Gauss equation
start_time = time.perf_counter()

if dx != dy:
    raise ValueError("dx and dy must be equal for this implementation.")

def boundary_conditions(nx, ny):     # interval x, interval y, u(x,y)
    """Set up boundary conditions for the potential field."""
    u = np.zeros((ny, nx))  # y and x are reversed due to the array indexing;

    # Remeber the coorinate of y is up-side down
    u[0, :] = bottom_boundary       # real bottom boundary
    u[-1, :] = top_boundary         # real top boundary
    u[:, 0] = left_boundary         # left boundary
    u[:, -1] = right_boundary       # right boundary

    return u

def Jacobi(u, max_iterations, rho=False):
    """Perform Jacobi iterations to solve the Laplace equation."""
    #ny, nx = u.shape
    u_new = u.copy()
    for iteration in trange(max_iterations, desc="Jacobi", unit="iter"):
        if rho is False:
            u_new[1:-1, 1:-1] = 0.25 * (u[2:, 1:-1] +u[:-2, 1:-1] +u[1:-1, 2:] +u[1:-1, :-2])
        else:
            u_new[1:-1, 1:-1] = 0.25 * ((u[2:, 1:-1] +u[:-2, 1:-1] +u[1:-1, 2:] +u[1:-1, :-2]) + (dx**2) * rho[1:-1, 1:-1]) #/ epsilon_0)   # requiring dx = dy, charge to epsilon ratio instead.

        # mimic the periodic boundary condition
        # u_new[1:-1, 0] = 0.25 * (u[2:, 0] + u[:-2, 0] + u[1:-1, 1] + u[1:-1, -1]) 
        # u_new[1:-1, -1] = 0.25 * (u[2:, -1] + u[:-2, -1] + u[1:-1, 0] + u[1:-1, -2])  

        #u_new[0:ordery_b1, orderx_b0:orderx_b1] = 0     # building block boundary condition

        u = u_new
    print(f'Final iteration: {iteration}, Residual: {np.linalg.norm(u_new - u)}')

    return u

def main():
    nx = int(width / dx + 1)
    ny = int(height / dy + 1)
    print (nx, ny) 
    v = boundary_conditions(nx, ny)
    rho = charge_density(nx, ny, point_charges_positions, charges, charge_radius)
    v = Jacobi(v, iterations, rho)    # u, max_iterations, rho, Gaussian unit per electron charge
    #v = Jacobi(v, iterations)

    x = np.linspace(x0, x0 + width, nx)
    y = np.linspace(y0, y0 + height, ny)
    X, Y = np.meshgrid(x, y)

    # Graph of electric potential
    plt.contourf(X, Y, v, levels=50, cmap='viridis')
    plt.colorbar(format='%.3g')
    plt.title('v')
    plt.xlabel('x')
    plt.ylabel('y')
    # plt.savefig("2point_charges-Jacobi.png", dpi=600)
    plt.show()
    plt.close()

    structure_name = "2point_charges-Jacobi.csv"
    with open(structure_name, "w") as f:
        f.write("v, x, y\n")
        for i in range(int(ny)):
            for j in range(int(nx)):
                f.write(f"{v[i, j]}, {x[j]}, {y[i]}\n")

    # Graph of charge density
    if rho is not None:
        plt.contourf(X, Y, rho, levels=50, cmap='viridis')
        plt.colorbar(format='%.3g')
        plt.title('rho')
        plt.xlabel('x')
        plt.ylabel('y')
        # plt.savefig("2point_charges-rho.png", dpi=600)
        plt.show()
        plt.close()

        charges_name = "2point_charges-rho.csv"
        with open(charges_name, "w") as f:
            f.write("rho, x, y\n")
            for i in range(int(ny)):
                for j in range(int(nx)):
                    f.write(f"{rho[i, j]}, {x[j]}, {y[i]}\n")



if __name__ == "__main__":
    main()